# LimeSurvey Analysis Notebook

This notebook loads and explores LimeSurvey exports.

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')

In [23]:
# Update filename if needed
csv_path = 'results-survey432293.csv'
df = pd.read_csv(csv_path)

print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

Rows: 11, Columns: 135


,id,submitdate,lastpage,startlanguage,seed,startdate,datestamp,Q00001[SQ001],Q00001[SQ002],Q00001[SQ003],...,groupTime5717,Q00030Time,Q00031Time,Q00032Time,Q00033Time,Q00034Time,groupTime5708,Q00035Time,Q00036Time,Q00037Time
0,2,2026-03-25 11:03:14,9,en,1851706079,2026-03-25 10:44:51,2026-03-25 11:03:14,No,Yes,No,...,20.58,NaN,NaN,NaN,NaN,NaN,5.30,NaN,NaN,NaN
1,4,2026-03-25 11:06:59,9,en,663332136,2026-03-25 10:45:13,2026-03-25 11:06:59,No,Yes,No,...,36.65,NaN,NaN,NaN,NaN,NaN,64.01,NaN,NaN,NaN
2,6,2026-03-25 11:04:24,9,en,386207750,2026-03-25 10:45:17,2026-03-25 11:04:24,No,Yes,No,...,35.19,NaN,NaN,NaN,NaN,NaN,3.16,NaN,NaN,NaN
3,7,2026-03-25 11:02:06,9,en,1234118210,2026-03-25 10:45:43,2026-03-25 11:02:06,No,No,No,...,15.19,NaN,NaN,NaN,NaN,NaN,56.62,NaN,NaN,NaN
4,8,2026-03-25 11:05:12,9,en,1444895015,2026-03-25 10:46:41,2026-03-25 11:05:12,No,Yes,No,...,26.27,NaN,NaN,NaN,NaN,NaN,72.29,NaN,NaN,NaN


In [24]:
# Missing values overview
df.isna().mean().sort_values(ascending=False).head(20)

Q00019Time    1.0
Q00016Time    1.0
Q00026        1.0
Q00014        1.0
Q00008        1.0
Q00009        1.0
Q00007        1.0
Q00012Time    1.0
Q00013Time    1.0
Q00014Time    1.0
Q00015Time    1.0
Q00017Time    1.0
Q00008Time    1.0
Q00007Time    1.0
Q00018Time    1.0
Q00023Time    1.0
Q00022Time    1.0
Q00020Time    1.0
Q00021Time    1.0
Q00024Time    1.0
dtype: float64

In [25]:
# Numeric summary
df.describe(include=[np.number]).T

,count,mean,std,min,25%,50%,75%,max
id,11.0,1.427273e+01,9.456119e+00,2.000000e+00,6.500000e+00,1.200000e+01,2.250000e+01,2.700000e+01
lastpage,11.0,9.000000e+00,0.000000e+00,9.000000e+00,9.000000e+00,9.000000e+00,9.000000e+00,9.000000e+00
seed,11.0,1.275994e+09,6.206947e+08,3.862078e+08,6.957050e+08,1.444895e+09,1.827729e+09,2.095281e+09
Q00007,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q00008,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
Q00034Time,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
groupTime5708,11.0,1.904109e+02,2.964741e+02,3.160000e+00,7.880000e+00,6.401000e+01,2.192900e+02,9.499700e+02
Q00035Time,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Q00036Time,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
# change the index to the id column

df = df.set_index("id")

In [ ]:
# drop all columns starting with Q00007..Q00012
# (intro questions + object comprehension) no need for that

prefixes = [f"Q{i:05d}" for i in range(7, 13)]
#text field
prefixes.append("Q00026")
prefixes.append("Q00014")
prefixes.append("Q00030")


cols_to_drop = [
    col for col in df.columns
    if any(col.startswith(prefix) for prefix in prefixes)
]
print(cols_to_drop)
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns")

['Q00007', 'Q00008', 'Q00009', 'Q00010', 'Q00011[SQ001]', 'Q00011[SQ002]', 'Q00011[SQ003]', 'Q00011[SQ004]', 'Q00011[SQ005]', 'Q00012[SQ001]', 'Q00012[SQ002]', 'Q00012[SQ003]', 'Q00012[SQ004]', 'Q00012[SQ005]', 'Q00012[SQ006]', 'Q00012[SQ007]', 'Q00014', 'Q00026', 'Q00030', 'Q00007Time', 'Q00008Time', 'Q00009Time', 'Q00010Time', 'Q00011Time', 'Q00012Time', 'Q00014Time', 'Q00026Time', 'Q00030Time']
Dropped 28 columns


In [29]:
# Question-code pairs for pre/post comparison

mental_model_pairs = {
    "saliency_safe_state": {"pre": "Q00012", "post": "Q00022"},
    "saliency_conflict_state": {"pre": "Q00013", "post": "Q00021"},
    "moe": {"pre": "Q00015", "post": "Q00024"},
    "action_heatmaps": {"pre": "Q00016", "post": "Q00025"},
}

explanation_trust_pairs = {
    "trust_item_1": {"pre": "Q00017", "post": "Q00031"},
    "trust_item_2": {"pre": "Q00018", "post": "Q00032"},
    "trust_item_3": {"pre": "Q00019", "post": "Q00033"},
    "trust_item_4": {"pre": "Q00020", "post": "Q00034"},
}

question_pairs = {
    "mental_models": mental_model_pairs,
    "explanation_trust": explanation_trust_pairs,
}

question_pairs

{'mental_models': {'saliency_safe_state': {'pre': 'Q00012', 'post': 'Q00022'},
  'saliency_conflict_state': {'pre': 'Q00013', 'post': 'Q00021'},
  'moe': {'pre': 'Q00015', 'post': 'Q00024'},
  'action_heatmaps': {'pre': 'Q00016', 'post': 'Q00025'}},
 'explanation_trust': {'trust_item_1': {'pre': 'Q00017', 'post': 'Q00031'},
  'trust_item_2': {'pre': 'Q00018', 'post': 'Q00032'},
  'trust_item_3': {'pre': 'Q00019', 'post': 'Q00033'},
  'trust_item_4': {'pre': 'Q00020', 'post': 'Q00034'}}}